In [7]:
import polars as pl
from typing import NamedTuple


class TestCase(NamedTuple):
    """Represents a test case with inputs and description."""
    base: int | float
    exponent: int | float
    expected_result: int | float



def run_power_test(base: int, exp: int, expected_result: int) -> pl.DataFrame:
    """
    Test pow() across different type combinations.
    
    This function exposes a bug where Int8 base values return 0 on overflow
    instead of wrapping or raising an error.
    
    Args:
        base: The base value to test
        exp: The exponent value to test
        
    Returns:
        A DataFrame with test metadata and results across different type combinations
    """
    df = pl.DataFrame({"base": [base], "exp": [exp]})

    # Literal Base
    lit_base = pl.lit(base)
    lit_base_int8 = pl.lit(base, dtype=pl.Int8)
    lit_base_int16 = pl.lit(base, dtype=pl.Int16)

    # Column Base
    col_base = pl.col("base")
    col_base_int8 = pl.col("base").cast(pl.Int8)
    col_base_int16 = pl.col("base").cast(pl.Int16)

    # Literal Exponent
    lit_exp = pl.lit(exp)
    lit_exp_int8 = pl.lit(exp, dtype=pl.Int8)
    lit_exp_int16 = pl.lit(exp, dtype=pl.Int16)

    # Column Exponent
    col_exp = pl.col("exp")
    col_exp_int8 = pl.col("exp").cast(pl.Int8)
    col_exp_int16 = pl.col("exp").cast(pl.Int16)

    result = df.select(
        pl.lit(base).alias("base_input"),
        pl.lit(exp).alias("exp_input"),
        pl.lit(int(base) ** int(exp)).alias("expected_result"),
        # Int8 cases - BUGGY: silently return 0 on overflow
        lit_base_int8.pow(col_exp).alias("int8_lit_x_col_exp"),
        lit_base_int8.pow(col_exp_int8).alias("int8_lit_x_col_exp_int8"),
        lit_base_int8.pow(col_exp_int16).alias("int8_lit_x_col_exp_int16"),
        col_base_int8.pow(lit_exp).alias("int8_col_x_lit_exp"),
        col_base_int8.pow(lit_exp_int8).alias("int8_col_x_lit_exp_int8"),
        col_base_int8.pow(lit_exp_int16).alias("int8_col_x_lit_exp_int16"),
    )

    return result



def run_all_tests() -> pl.DataFrame:
    """
    Run all test cases and stack results into a single DataFrame.
    
    Returns:
        A DataFrame with all test results, one row per test case
    """
    test_cases = [
        TestCase(2, 8, 2**8),
        TestCase(2, 10, 2**10),
        TestCase(127, 2, 127**2),
        TestCase(10, 3,  10**3),
        TestCase(3, 5,  3**5),
        TestCase(5, 4,  5**4),
        TestCase(1, 127,  1**127),
        TestCase(2, 7,  2**7),
        TestCase(2, 6,  2**6),
    ]

    results = []
    for test in test_cases:
        df = run_power_test(test.base, test.exponent, test.expected_result)
        results.append(df)

    # Stack all results
    combined = pl.concat(results)
    return combined
    
run_all_tests()


base_input,exp_input,expected_result,int8_lit_x_col_exp,int8_lit_x_col_exp_int8,int8_lit_x_col_exp_int16,int8_col_x_lit_exp,int8_col_x_lit_exp_int8,int8_col_x_lit_exp_int16
i32,i32,i32,i8,i8,i8,i8,i8,i8
2,8,256,0,0,0,0,0,0
2,10,1024,0,0,0,0,0,0
127,2,16129,1,1,1,1,1,1
10,3,1000,-24,-24,-24,-24,-24,-24
3,5,243,-13,-13,-13,-13,-13,-13
5,4,625,113,113,113,113,113,113
1,127,1,1,1,1,1,1,1
2,7,128,-128,-128,-128,-128,-128,-128
2,6,64,64,64,64,64,64,64


In [32]:
import polars as pl

df = pl.DataFrame({'base': [2], 'exp': [8]})

# Expression definitions
lit_base = pl.lit(2)

# Base as literal or column - forced to Int8
lit_base_i8 = pl.lit(2, dtype=pl.Int8)
col_base_i8 = pl.col('base').cast(pl.Int8)

# Exponent as literal or column
lit_exp = pl.lit(8)
col_exp = pl.col('exp')

# Execution: Int8 base - silently returns 0 (should be 256)
results = df.select(
    #Correct Case - 256
    lit_base.pow(col_exp).alias("lit_base**col_exp"),
    #Failure Cases - 0
    lit_base_i8.pow(col_exp).alias("lit_base_i8**col_exp"),
    col_base_i8.pow(lit_exp).alias("col_base_i8**lit_exp"),
    lit_base_i8.pow(lit_exp).alias("lit_base_i8**lit_exp"),
    col_base_i8.pow(col_exp).alias("col_base_i8**col_exp"),

    
).unpivot(variable_name="case", value_name="result")
print(results.to_pandas().to_markdown(index=False))

| case                 |   result |
|:---------------------|---------:|
| lit_base**col_exp    |      256 |
| lit_base_i8**col_exp |        0 |
| col_base_i8**lit_exp |        0 |
| lit_base_i8**lit_exp |        0 |
| col_base_i8**col_exp |        0 |


In [26]:
z = lambda: results
z()

case,result
str,i32
"""lit_base**col_exp""",256
"""lit_base_i8**col_exp""",0
"""col_base_i8**lit_exp""",0
"""lit_base_i8**lit_exp""",0
"""col_base_i8**col_exp""",0


In [31]:
print(results.to_pandas().to_markdown(index=False))

| case                 |   result |
|:---------------------|---------:|
| lit_base**col_exp    |      256 |
| lit_base_i8**col_exp |        0 |
| col_base_i8**lit_exp |        0 |
| lit_base_i8**lit_exp |        0 |
| col_base_i8**col_exp |        0 |


In [1]:
import ibis
import ibis.selectors as s
import polars as pl
import pandas as pd

def test_polars_backend():
    """Test with Polars backend"""
    print(f"\n{'='*60}")
    print(f"Testing with Polars backend")
    print(f"{'='*60}")
    
    ibis.set_backend('polars')
    polars_backend = ibis.polars.connect()
    sqlite_backend = ibis.sqlite.connect()
    duckdb_backend = ibis.duckdb.connect()
    
    # Create base table
    pl_df = pl.DataFrame({'base': [2], 'exp': [8]})
    
    #ibis backend tables
    df_polars = polars_backend.create_table('df_polars', pl_df)
    df_sqlite = sqlite_backend.create_table('pl_df', pl_df)
    df_duckdb = duckdb_backend.create_table('pl_df', pl_df)
    
    # Expression definitions
    lit_base = ibis.literal(2)
    col_base = df.base
   
    lit_exp = ibis.literal(8)
    col_exp = df.exp
    
    # Build the results
    polars_results = df_polars.select(
        ibis.literal("polars").name("backend"),
        lit_base.pow(col_exp).name("lit_base**col_exp"),
        col_base.pow(lit_exp).name("col_base**lit_exp"),
        lit_base.pow(lit_exp).name("lit_base**lit_exp"),
        col_base.pow(col_exp).name("col_base**col_exp"),
    )


    
    # Convert to long format
    results_unpivot = results.pivot_longer(
        s.all(),
        names_to='case',
        values_to='result'
    )
    
    pandas_df = results_unpivot.execute()
    print(pandas_df.to_markdown(index=False))
    return pandas_df


def test_sqlite_backend():
    """Test with SQLite backend"""
    print(f"\n{'='*60}")
    print(f"Testing with SQLite backend")
    print(f"{'='*60}")
    
    backend = ibis.sqlite.connect()
    
    # Create base table
    pl_df = pl.DataFrame({'base': [2], 'exp': [8]})
    df = backend.create_table('power_test', pl_df, temp=True)
    
    # Expression definitions - SQLite doesn't have explicit int8
    lit_base = ibis.literal(2)
    lit_base_i8 = ibis.literal(2)  # SQLite uses INTEGER type
    col_base_i8 = df.base
    
    lit_exp = ibis.literal(8)
    col_exp = df.exp
    
    # Build results - manually unpivot since pivot_longer may not work well
    result_list = []
    
    cases = [
        ("lit_base**col_exp", lit_base.pow(col_exp)),
        ("lit_base_i8**col_exp", lit_base_i8.pow(col_exp)),
        ("col_base_i8**lit_exp", col_base_i8.pow(lit_exp)),
        ("lit_base_i8**lit_exp", lit_base_i8.pow(lit_exp)),
        ("col_base_i8**col_exp", col_base_i8.pow(col_exp)),
    ]
    
    for case_name, expr in cases:
        result = df.select(result=expr).execute()
        result_list.append({'case': case_name, 'result': result['result'].iloc[0]})
    
    pandas_df = pd.DataFrame(result_list)
    print(pandas_df.to_markdown(index=False))
    return pandas_df


def test_duckdb_backend():
    """Test with DuckDB backend"""
    print(f"\n{'='*60}")
    print(f"Testing with DuckDB backend")
    print(f"{'='*60}")
    
    backend = ibis.duckdb.connect()
    
    # Create base table
    pl_df = pl.DataFrame({'base': [2], 'exp': [8]})
    df = backend.create_table('power_test', pl_df)
    
    # Expression definitions
    lit_base = ibis.literal(2)
    lit_base_i8 = ibis.literal(2, type='int8')
    col_base_i8 = df.base.cast('int8')
    
    lit_exp = ibis.literal(8)
    col_exp = df.exp
    
    # Build results - manually unpivot
    result_list = []
    
    cases = [
        ("lit_base**col_exp", lit_base.pow(col_exp)),
        ("lit_base_i8**col_exp", lit_base_i8.pow(col_exp)),
        ("col_base_i8**lit_exp", col_base_i8.pow(lit_exp)),
        ("lit_base_i8**lit_exp", lit_base_i8.pow(lit_exp)),
        ("col_base_i8**col_exp", col_base_i8.pow(col_exp)),
    ]
    
    for case_name, expr in cases:
        result = df.select(result=expr).execute()
        result_list.append({'case': case_name, 'result': result['result'].iloc[0]})
    
    pandas_df = pd.DataFrame(result_list)
    print(pandas_df.to_markdown(index=False))
    return pandas_df


# Run all tests
polars_results = test_polars_backend()
sqlite_results = test_sqlite_backend()
duckdb_results = test_duckdb_backend()

print("\n" + "="*60)
print("Summary: All backends tested")
print("="*60)

AttributeError: module 'ibis' has no attribute 'DataFrame'. 

In [16]:
import ibis
import ibis.selectors as s
import polars as pl
import pandas as pd

    
ibis.set_backend('polars')
polars_backend = ibis.polars.connect()
sqlite_backend = ibis.sqlite.connect()
duckdb_backend = ibis.duckdb.connect()

# Create base table
pl_df = pl.DataFrame({'base': [2], 'exp': [8]})

#ibis backend tables
df_polars = polars_backend.create_table('df_polars', pl_df)
df_sqlite = sqlite_backend.create_table('pl_df', pl_df)
df_duckdb = duckdb_backend.create_table('pl_df', pl_df)

# Expression definitions
lit_base = ibis.literal(2)
lit_exp = ibis.literal(8)

# Build the results
# Polars - will fail with literals on the LHS as they are cast to Int8 
polars_results = df_polars.select(
    ibis.literal("polars").name("backend"),
    lit_base.pow(df_polars.exp).name("lit_base**col_exp"),
    df_polars.base.pow(lit_exp).name("col_base**lit_exp"),
    lit_base.pow(lit_exp).name("lit_base**lit_exp"),
    df_polars.base.pow(df_polars.exp).name("col_base**col_exp"),
).execute()

# SQLite - all correct
sqlite_results = df_sqlite.select(
    ibis.literal("sqlite").name("backend"),
    lit_base.pow(df_sqlite.exp).name("lit_base**col_exp"),
    df_sqlite.base.pow(lit_exp).name("col_base**lit_exp"),
    lit_base.pow(lit_exp).name("lit_base**lit_exp"),
    df_sqlite.base.pow(df_sqlite.exp).name("col_base**col_exp"),
).execute()

# DuckDB - all correct
duckdb_results = df_duckdb.select(
    ibis.literal("duckdb").name("backend"),
    lit_base.pow(df_duckdb.exp).name("lit_base**col_exp"),
    df_duckdb.base.pow(lit_exp).name("col_base**lit_exp"),
    lit_base.pow(lit_exp).name("lit_base**lit_exp"),
    df_duckdb.base.pow(df_duckdb.exp).name("col_base**col_exp"),
).execute()

results = pd.concat([polars_results, duckdb_results, sqlite_results])

print(results.to_markdown(index=False))


| backend   |   lit_base**col_exp |   col_base**lit_exp |   lit_base**lit_exp |   col_base**col_exp |
|:----------|--------------------:|--------------------:|--------------------:|--------------------:|
| polars    |                   0 |                 256 |                   0 |                 256 |
| duckdb    |                 256 |                 256 |                 256 |                 256 |
| sqlite    |                 256 |                 256 |                 256 |                 256 |


In [ ]:
| backend   |   lit_base**col_exp |   col_base**lit_exp |   lit_base**lit_exp |   col_base**col_exp |
|:----------|--------------------:|--------------------:|--------------------:|--------------------:|
| polars    |                   0 |                 256 |                   0 |                 256 |
| duckdb    |                 256 |                 256 |                 256 |                 256 |
| sqlite    |                 256 |                 256 |                 256 |                 256 |